## Example 1: Basic Encoding Lookup and Standards

In [ ]:
import webencodings

# Example 1: Basic encoding lookup and web standard compliance
print("Web encoding lookup and standards compliance:")

# Test various encoding labels as they appear in web contexts
web_encoding_labels = [
    "utf-8",
    "UTF-8", 
    "iso-8859-1",
    "latin1",
    "ascii",
    "windows-1252",
    "cp1252",
    "us-ascii",
    "iso8859-1"
]

print("Encoding label normalization:")
for label in web_encoding_labels:
    encoding = webencodings.lookup(label)
    if encoding:
        print(f"'{label}' → {encoding.name}")
    else:
        print(f"'{label}' → Not found")

# Show available encoding objects
print(f"\nEncoding object properties:")
utf8_encoding = webencodings.lookup('utf-8')
print(f"UTF-8 encoding:")
print(f"  Name: {utf8_encoding.name}")
print(f"  Codec info: {utf8_encoding.codec_info}")

# Test labels iteration
print(f"\nSample of supported labels:")
label_count = 0
seen_encodings = set()
for label in list(webencodings.LABELS.keys())[:15]:
    encoding = webencodings.lookup(label)
    if encoding and encoding.name not in seen_encodings:
        print(f"  '{label}' → {encoding.name}")
        seen_encodings.add(encoding.name)
        label_count += 1
    if label_count >= 10:
        break

print(f"\nTotal supported labels: {len(webencodings.LABELS)}")
print("Webencodings follows the WHATWG Encoding Standard")

## Example 2: Encoding and Decoding Web Content

In [ ]:
import webencodings

# Example 2: Encoding and decoding web content
print("Web content encoding and decoding:")

# Test text with various characters
test_texts = [
    "Hello, World!",
    "Café München résumé",
    "中文 日本語 한글",
    "العربية עברית русский",
    "Emoji content"
]

# Test encoding to bytes
print("Encoding text to bytes:")
for text in test_texts:
    # Encode using webencodings (returns just bytes)
    encoded_bytes = webencodings.encode(text, 'utf-8')
    encoding_obj = webencodings.lookup('utf-8')
    print(f"'{text}' → {len(encoded_bytes)} bytes (encoding: {encoding_obj.name})")

# Test decoding from bytes
print(f"\nDecoding bytes to text:")
for text in test_texts:
    # Encode first, then decode
    encoded_bytes = webencodings.encode(text, 'utf-8')
    decoded_text, encoding_used = webencodings.decode(encoded_bytes, 'utf-8')
    
    round_trip_success = text == decoded_text
    print(f"Round-trip: {'✓' if round_trip_success else '✗'} '{text}'")

# Test different encodings
print(f"\nTesting different encodings:")
test_text = "Café naïve résumé"
encodings_to_test = ['utf-8', 'windows-1252', 'iso-8859-1']

for encoding_name in encodings_to_test:
    try:
        encoded_bytes = webencodings.encode(test_text, encoding_name)
        decoded_text, _ = webencodings.decode(encoded_bytes, encoding_name)
        
        print(f"{encoding_name:12}: {len(encoded_bytes)} bytes, round-trip: {'✓' if test_text == decoded_text else '✗'}")
    except Exception as e:
        print(f"{encoding_name:12}: Error - {e}")
        raise

# Test incremental encoding/decoding
print(f"\nIncremental processing:")
text_chunks = ["Hello ", "世界", "! How ", "are you?"]

# Incremental encoding
encoder = webencodings.IncrementalEncoder('utf-8')
encoded_chunks = []
for chunk in text_chunks:
    encoded_chunk = encoder.encode(chunk)
    encoded_chunks.append(encoded_chunk)
final_encoded = encoder.encode('', final=True)
if final_encoded:
    encoded_chunks.append(final_encoded)

print(f"Incremental encoding: {len(encoded_chunks)} chunks")

# Incremental decoding
decoder = webencodings.IncrementalDecoder('utf-8')
decoded_result = ""
for chunk in encoded_chunks:
    decoded_chunk = decoder.decode(chunk)
    decoded_result += decoded_chunk

original_text = "".join(text_chunks)
print(f"Incremental result: {'✓' if original_text == decoded_result else '✗'}")
print(f"Original: '{original_text}'")
print(f"Decoded:  '{decoded_result}'")

## Example 3: Streaming and Iterative Processing

In [ ]:
import webencodings

# Example 3: Iterative processing and streaming
print("Iterative processing and streaming:")

# Test iter_encode for streaming encoding
test_text = "This is a longer text that will be processed in chunks for streaming encoding. It contains various characters: café, naïve, résumé"

print("Streaming encoding (iter_encode):")
chunk_count = 0
total_bytes = 0

# iter_encode returns a generator of byte chunks
for chunk in webencodings.iter_encode(test_text, 'utf-8'):
    chunk_count += 1
    total_bytes += len(chunk)
    if chunk_count <= 5:  # Show first few chunks
        print(f"Chunk {chunk_count}: {len(chunk)} bytes - {chunk[:20]}")

print(f"Total bytes processed: {total_bytes}")

# Test iter_decode for streaming decoding
print(f"\nStreaming decoding (iter_decode):")

# Create some byte chunks to decode
encoded_bytes = webencodings.encode(test_text, 'utf-8')
chunk_size = 10
byte_chunks = [encoded_bytes[i:i+chunk_size] for i in range(0, len(encoded_bytes), chunk_size)]

# iter_decode returns a tuple: (generator, encoding)
decoded_generator, encoding_used = webencodings.iter_decode(byte_chunks, 'utf-8')
print(f"Decoding with encoding: {encoding_used.name}")

decoded_parts = []
chunk_count = 0

for decoded_chunk in decoded_generator:
    chunk_count += 1
    decoded_parts.append(decoded_chunk)
    if chunk_count <= 5:  # Show first few chunks
        print(f"Decoded chunk {chunk_count}: '{decoded_chunk[:20]}{'...' if len(decoded_chunk) > 20 else ''}'")

# Verify streaming decode
full_decoded = "".join(decoded_parts)
print(f"Total decoded chunks: {chunk_count}")
print(f"Streaming decode success: {'✓' if full_decoded == test_text else '✗'}")

# Test different chunk sizes for streaming
print(f"\nStreaming with different chunk sizes:")
chunk_sizes = [5, 10, 20, 50]

for chunk_size in chunk_sizes:
    byte_chunks = [encoded_bytes[i:i+chunk_size] for i in range(0, len(encoded_bytes), chunk_size)]
    
    # Count chunks and decode
    decoded_generator, _ = webencodings.iter_decode(byte_chunks, 'utf-8')
    decoded_text = ""
    chunk_count = 0
    for decoded_chunk in decoded_generator:
        decoded_text += decoded_chunk
        chunk_count += 1
    
    success = decoded_text == test_text
    print(f"Chunk size {chunk_size:2d}: {chunk_count:2d} chunks, success: {'✓' if success else '✗'}")

# Demonstrate handling of incomplete byte sequences
print(f"\nHandling incomplete byte sequences:")
utf8_text = "Café 世界"
full_bytes = webencodings.encode(utf8_text, 'utf-8')

# Split in the middle of a multi-byte character
split_point = 6  # This should split a multi-byte UTF-8 sequence
incomplete_chunks = [full_bytes[:split_point], full_bytes[split_point:]]

print(f"Original text: '{utf8_text}'")
print(f"Byte sequence split at position {split_point}")

try:
    decoded_generator, _ = webencodings.iter_decode(incomplete_chunks, 'utf-8')
    decoded_text = ""
    for chunk in decoded_generator:
        decoded_text += chunk
    
    print(f"Decoded result: '{decoded_text}'")
    print(f"Handles incomplete sequences: {'✓' if decoded_text == utf8_text else '✗'}")
except Exception as e:
    print(f"Error handling incomplete sequences: {e}")
    raise

print(f"\nStreaming enables efficient processing of large web content")
print("Essential for handling HTTP responses and file uploads")

## Example 4: Web Standards Compliance and Detection

In [ ]:
import webencodings

# Example 4: Web standards compliance and encoding detection
print("Web standards compliance and encoding detection:")

# Test HTML document encoding scenarios
html_samples = [
    ('<!DOCTYPE html><html><head><meta charset="utf-8"></head><body>Hello 世界</body></html>', 'utf-8'),
    ('<!DOCTYPE html><html><head><meta charset="iso-8859-1"></head><body>Café</body></html>', 'iso-8859-1'),
    ('<!DOCTYPE html><html><head><meta charset="windows-1252"></head><body>naïve résumé</body></html>', 'windows-1252'),
    ('<!DOCTYPE html><html><head><meta http-equiv="content-type" content="text/html; charset=utf-8"></head><body>Test</body></html>', 'utf-8')
]

print("HTML encoding declaration testing:")
for html_content, declared_encoding in html_samples:
    # Extract the content part for encoding
    body_start = html_content.find('<body>') + 6
    body_end = html_content.find('</body>')
    content = html_content[body_start:body_end]
    
    # Test encoding with declared encoding
    try:
        encoded_bytes = webencodings.encode(content, declared_encoding)
        decoded_content, _ = webencodings.decode(encoded_bytes, declared_encoding)
        
        success = content == decoded_content
        print(f"'{declared_encoding}': {len(encoded_bytes)} bytes, success: {'✓' if success else '✗'}")
        print(f"  Content: '{content}' → '{decoded_content}'")
    except Exception as e:
        print(f"'{declared_encoding}': Error - {e}")
        raise

# Test WHATWG encoding standard compliance
print(f"\nWHATWG Encoding Standard compliance:")

# Test label aliases that should resolve to the same encoding
label_groups = [
    ['utf-8', 'UTF-8', 'utf8', 'unicode-1-1-utf-8'],
    ['windows-1252', 'cp1252', 'x-cp1252'],
    ['iso-8859-1', 'latin1', 'iso8859-1', 'l1']
]

for group in label_groups:
    print(f"\nLabel group: {group}")
    encodings = [webencodings.lookup(label) for label in group]
    
    # Check if all labels resolve to the same encoding
    if all(enc and enc.name == encodings[0].name for enc in encodings):
        print(f"  ✓ All resolve to: {encodings[0].name}")
    else:
        print(f"  ✗ Mixed results: {[enc.name if enc else None for enc in encodings]}")

# Test encoding fallbacks and replacements
print(f"\nEncoding fallbacks and error handling:")

# Text that might not be representable in all encodings. WHATWG maps the 'ascii' and 'iso-8859-1'
# labels to windows-1252, so every target but utf-8 is single-byte: each sample names where it fits.
all_targets = {'ascii', 'windows-1252', 'iso-8859-1', 'utf-8'}
test_texts = [
    ("ASCII safe", "Hello World", all_targets),
    ("Latin extended", "Café naïve résumé", all_targets),
    ("Non-Latin", "中文 русский", {'utf-8'}),
    ("Special chars", "Hello World test", all_targets)
]

target_encodings = ['ascii', 'windows-1252', 'iso-8859-1', 'utf-8']

for desc, text, representable_in in test_texts:
    print(f"\n{desc}: '{text}'")
    for encoding_name in target_encodings:
        if encoding_name in representable_in:
            encoded_bytes = webencodings.encode(text, encoding_name)
            decoded_text, _ = webencodings.decode(encoded_bytes, encoding_name)
            
            success = text == decoded_text
            print(f"  {encoding_name:12}: {'✓' if success else '✗'} ({len(encoded_bytes)} bytes)")
            assert success, f"{text!r} did not round-trip through {encoding_name}"
        else:
            try:
                webencodings.encode(text, encoding_name)
            except UnicodeEncodeError as e:
                print(f"  {encoding_name:12}: not representable, as expected ({e.reason})")
            else:
                raise AssertionError(f"{text!r} unexpectedly encoded in {encoding_name}")

# Test BOM (Byte Order Mark) handling
print(f"\nBOM (Byte Order Mark) handling:")

utf8_with_bom = b'\xef\xbb\xbf' + "Hello 世界".encode('utf-8')
utf8_without_bom = "Hello 世界".encode('utf-8')

print(f"UTF-8 with BOM: {len(utf8_with_bom)} bytes")
print(f"UTF-8 without BOM: {len(utf8_without_bom)} bytes")

# Decode both versions
decoded_with_bom, _ = webencodings.decode(utf8_with_bom, 'utf-8')
decoded_without_bom, _ = webencodings.decode(utf8_without_bom, 'utf-8')

print(f"Decoded with BOM: '{decoded_with_bom}'")
print(f"Decoded without BOM: '{decoded_without_bom}'")

print(f"\nWebencodings ensures web standard compliance")
print("Critical for reliable web content processing")

## Example 5: Production Web Applications

In [ ]:
import webencodings
import io

# Example 5: Production patterns for web applications
class WebContentProcessor:
    """Production-ready web content encoding processor"""
    
    def __init__(self):
        self.stats = {
            "documents_processed": 0,
            "encoding_errors": 0,
            "fallback_used": 0,
            "bytes_processed": 0
        }
        self.default_encoding = 'utf-8'
        self.fallback_encodings = ['windows-1252', 'iso-8859-1']
    
    def detect_and_decode(self, byte_content, declared_encoding=None):
        """Detect encoding and decode content with fallback support"""
        self.stats["documents_processed"] += 1
        self.stats["bytes_processed"] += len(byte_content)
        
        # Priority list: declared encoding, then fallbacks
        encodings_to_try = []
        if declared_encoding:
            encodings_to_try.append(declared_encoding)
        encodings_to_try.append(self.default_encoding)
        encodings_to_try.extend(self.fallback_encodings)
        
        # Remove duplicates while preserving order
        seen = set()
        encodings_to_try = [enc for enc in encodings_to_try 
                           if not (enc in seen or seen.add(enc))]
        
        for encoding_name in encodings_to_try:
            try:
                encoding = webencodings.lookup(encoding_name)
                if encoding:
                    decoded_text, actual_encoding = webencodings.decode(byte_content, encoding_name)
                    
                    # Track if we used fallback
                    if encoding_name != (declared_encoding or self.default_encoding):
                        self.stats["fallback_used"] += 1
                    
                    return {
                        "text": decoded_text,
                        "encoding": actual_encoding.name,
                        "declared": declared_encoding,
                        "success": True
                    }
            except Exception as e:
                self.stats["encoding_errors"] += 1
                continue
        
        # Complete failure
        return {
            "text": None,
            "encoding": None,
            "declared": declared_encoding,
            "success": False,
            "error": "All encoding attempts failed"
        }
    
    def process_http_response(self, content_bytes, content_type_header=None):
        """Process HTTP response with Content-Type header parsing"""
        declared_encoding = None
        
        # Parse Content-Type header for charset
        if content_type_header:
            parts = content_type_header.lower().split(';')
            for part in parts:
                part = part.strip()
                if part.startswith('charset='):
                    declared_encoding = part.split('=', 1)[1].strip().strip('"\'')
                    break
        
        return self.detect_and_decode(content_bytes, declared_encoding)
    
    def stream_decode(self, byte_stream, encoding='utf-8', chunk_size=8192):
        """Stream decode large content efficiently"""
        decoder = webencodings.IncrementalDecoder(encoding)
        
        while True:
            chunk = byte_stream.read(chunk_size)
            if not chunk:
                # Final chunk
                final_text = decoder.decode(b'', final=True)
                if final_text:
                    yield final_text
                break
            
            decoded_chunk = decoder.decode(chunk)
            if decoded_chunk:
                yield decoded_chunk
    
    def normalize_for_storage(self, text_content):
        """Normalize text content for consistent storage"""
        # Always encode to UTF-8 for storage
        encoded_bytes = webencodings.encode(text_content, 'utf-8')
        
        # Verify round-trip
        decoded_text, _ = webencodings.decode(encoded_bytes, 'utf-8')
        
        return {
            "normalized_bytes": encoded_bytes,
            "encoding": "utf-8",
            "round_trip_success": text_content == decoded_text,
            "byte_count": len(encoded_bytes)
        }
    
    def get_performance_stats(self):
        """Get processing performance statistics"""
        stats = self.stats.copy()
        if stats["documents_processed"] > 0:
            stats["avg_bytes_per_doc"] = stats["bytes_processed"] / stats["documents_processed"]
            stats["error_rate"] = stats["encoding_errors"] / stats["documents_processed"]
            stats["fallback_rate"] = stats["fallback_used"] / stats["documents_processed"]
        return stats

print("Production web content processing:")

# Create processor
processor = WebContentProcessor()

# Test HTTP response scenarios
print("1. HTTP response processing:")
# (bytes, Content-Type, the text that was encoded, the encoding it must be decoded with)
http_scenarios = [
    (b"Hello, World!", "text/html; charset=utf-8", "Hello, World!", "utf-8"),
    ("Café naïve résumé".encode('windows-1252'), "text/html; charset=windows-1252", "Café naïve résumé", "windows-1252"),
    ("中文内容".encode('utf-8'), "text/html; charset=utf-8", "中文内容", "utf-8"),
    (b"Plain text", None, "Plain text", "utf-8"),  # No charset declared
]

for content_bytes, content_type, expected_text, expected_encoding in http_scenarios:
    result = processor.process_http_response(content_bytes, content_type)
    print(f"Content-Type: {content_type or 'None'}")
    print(f"  Success: {'✓' if result['success'] else '✗'}")
    # every scenario is decodable and must give back the encoded text: webencodings replaces undecodable bytes with
    # U+FFFD instead of raising, so `success` alone cannot catch a wrong charset
    assert result['success'], result
    assert (result['text'], result['encoding']) == (expected_text, expected_encoding), result
    print(f"  Text: '{result['text'][:30]}...' (encoding: {result['encoding']})")

# Test streaming decode
print(f"\n2. Streaming decode:")
large_content = "This is a large document with international content: " * 50 + "世界 café naïve"
large_bytes = large_content.encode('utf-8')
byte_stream = io.BytesIO(large_bytes)

decoded_chunks = []
chunk_count = 0
for chunk in processor.stream_decode(byte_stream, encoding='utf-8', chunk_size=100):
    decoded_chunks.append(chunk)
    chunk_count += 1
    if chunk_count <= 3:  # Show first few chunks
        print(f"  Chunk {chunk_count}: {len(chunk)} chars - '{chunk[:30]}...'")

full_decoded = "".join(decoded_chunks)
stream_success = full_decoded == large_content
print(f"  Streaming success: {'✓' if stream_success else '✗'} ({chunk_count} chunks)")
assert stream_success, "incremental decode of the chunked stream does not reproduce the document"

# Test normalization for storage
print(f"\n3. Content normalization:")
test_contents = [
    "Simple ASCII content",
    "Extended Latin: Café München naïve résumé",
    "Mixed scripts: Hello 世界 мир",
]

for content in test_contents:
    normalized = processor.normalize_for_storage(content)
    print(f"'{content[:30]}...':")
    print(f"  Bytes: {normalized['byte_count']}, Round-trip: {'✓' if normalized['round_trip_success'] else '✗'}")
    assert normalized['round_trip_success'], f"{content!r} did not round-trip through UTF-8 storage"

# Show performance statistics
print(f"\n4. Performance statistics:")
stats = processor.get_performance_stats()
for key, value in stats.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.3f}")
    else:
        print(f"  {key}: {value}")

print(f"\nProduction benefits:")
benefits = [
    "WHATWG Encoding Standard compliance",
    "Robust fallback handling for encoding errors",
    "Efficient streaming for large content",
    "HTTP header parsing integration",
    "Performance monitoring and metrics",
    "Consistent UTF-8 normalization for storage"
]

for benefit in benefits:
    print(f"  ✓ {benefit}")

print(f"\nWebencodings enables reliable web content processing")